# Approval Workflows

Autonomous AI agents can draft emails, generate reports, and propose actions at high speed - but in production, many of those actions carry real consequences once executed. A mis-phrased customer email, a financial commitment made without authority, or a policy exception granted without oversight can cause harm that is difficult to undo. The challenge is that agents lack the organizational context, legal awareness, and risk intuition that experienced humans bring to these decisions.

Approval workflows solve this by introducing deliberate checkpoints in the execution pipeline. Rather than acting immediately, the agent generates a proposal, suspends execution, and routes the proposal to a human reviewer. Only after an explicit approval does the action proceed. This notebook builds a practical approval workflow - a customer-service email agent that drafts communications but cannot send them without human sign-off. We cover the core approval loop, timeout handling, audit logging, and a multi-level approval hierarchy for higher-stakes actions.

In [1]:
import os
from datetime import datetime, timedelta
from typing import TypedDict, Literal, Optional, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

### Initialize the language model

In [2]:
# temperature=0.7 produces varied email drafts; set to 0 for fully deterministic output
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Understanding approval workflows
An approval workflow is not just a yes/no gate on top of an agent. It is a structured handoff between autonomous execution and human judgment, and the quality of that handoff determines whether the human can make a fast, well-informed decision. The core sequence is: the agent proposes an action with accompanying reasoning, the system captures the proposal in a "pending" state, a human reviewer sees the full context and decides, and the workflow branches based on that decision.

Three design principles distinguish effective approval workflows from bureaucratic ones. First, the agent must surface its reasoning alongside the proposal - reviewers who see only the output cannot efficiently judge whether the intent was appropriate. Second, the system must have a defined response to timeout: if a reviewer does not respond within a set window, the workflow should take a safe default action rather than waiting indefinitely. Third, every decision - including who decided, when, and why - should be written into an immutable audit log that supports both compliance reporting and retrospective analysis.

### Define the approval state
The state structure is what ties these three principles together. It holds all information flowing through the workflow - customer context, agent-generated content, approval tracking, and the audit trail - so that every node reads from and writes to a single, consistent source of truth.

In [3]:
class ApprovalState(TypedDict):
    # --- Customer context (set once at workflow start, never modified) ---
    customer_request: str          # The original request that triggered this workflow
    customer_email: str            # Destination address for the email

    # --- Agent-generated content (written by generate_email_draft) ---
    draft_subject: Optional[str]   # Proposed subject line
    draft_body: Optional[str]      # Proposed email body
    draft_reasoning: Optional[str] # Why the agent wrote it this way — shown to the reviewer

    # --- Approval tracking ---
    # Status lifecycle: None → "pending" → "approved" | "rejected" | "timeout"
    approval_status: Optional[Literal["pending", "approved", "rejected", "timeout"]]
    approval_requested_at: Optional[datetime]  # When the review window opened
    approval_decided_at: Optional[datetime]    # When the human responded
    approver_name: Optional[str]               # Who decided (for accountability)
    approval_notes: Optional[str]              # Reason provided by the reviewer
    timeout_minutes: int                       # Max minutes to wait before auto-timeout

    # --- Execution outcome (written by the terminal node) ---
    email_sent: bool
    final_status: Optional[str]

    # --- Audit trail (append-only; every node adds one entry) ---
    audit_log: List[dict]

The `ApprovalState` TypedDict groups fields into four logical zones, each owned by a specific part of the workflow:
- The customer context fields (`customer_request`, `customer_email`) are set once in the initial call and never modified by any node.
- The draft fields (`draft_subject`, `draft_body`, `draft_reasoning`) are written exclusively by the generation node.
- The approval tracking fields progress through a defined status lifecycle. The `approval_requested_at` timestamp is especially important - it is what the decision node compares against `timeout_minutes` to detect expired review windows.
- The execution fields (`email_sent`, `final_status`) are finalized by whichever terminal node runs last.

The `audit_log` list is append-only by convention: every node extends it as `state["audit_log"] + [new_entry]` rather than mutating in place. Storing `timeout_minutes` in state rather than hardcoding it allows different workflow invocations to use different thresholds - urgent requests might use a five-minute window while routine ones use sixty.

### Generate an email draft
The first node to implement is draft generation. The agent needs to produce not just the email content but also the reasoning behind its choices, because that reasoning is what allows a reviewer to evaluate the draft in seconds rather than minutes.

In [4]:
def generate_email_draft(state: ApprovalState) -> dict:
    """Generate an email draft and the reasoning behind it.

    Produces three clearly delimited sections — REASONING, SUBJECT, BODY —
    so downstream parsing is unambiguous even if the model adds extra text.
    """
    prompt = f"""You are a professional customer service email writer.

Customer request: {state['customer_request']}
Customer email: {state['customer_email']}

Write a response email using exactly these section headings:

REASONING:
Explain your tone choice, what key points you addressed, and any risks the reviewer should be aware of (e.g. commitments made, sensitive language).

SUBJECT:
A clear, professional subject line (one line only).

BODY:
The full email body, ready to send."""

    response = llm.invoke(prompt)
    content = response.content

    # Parse the three sections by splitting on the literal headers
    try:
        after_reasoning = content.split("SUBJECT:")
        reasoning = after_reasoning[0].replace("REASONING:", "").strip()

        after_subject = after_reasoning[1].split("BODY:")
        subject = after_subject[0].strip()
        body = after_subject[1].strip()

    except (IndexError, ValueError):
        # If the model doesn't follow the format, fall back to safe values
        # rather than crashing the entire workflow
        reasoning = "Could not parse agent reasoning."
        subject = "Customer Service Response"
        body = content.strip()

    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "draft_generated",
        "details": f"Subject: {subject[:60]}"
    }

    # Return only the fields this node owns; all other state fields remain unchanged
    return {
        "draft_subject": subject,
        "draft_body": body,
        "draft_reasoning": reasoning,
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

The `generate_email_draft` function uses a structured prompt that forces the LLM to produce three clearly delimited sections.
- The prompt labels each section with a literal header (`REASONING:`, `SUBJECT:`, `BODY:`) that the parsing logic splits on. This keeps the parse reliable while still allowing the LLM to write freely within each section.
- The `try/except` block around the parsing ensures the workflow does not crash if the LLM deviates from the expected format - safe fallback values are set and execution continues, which is preferable to a hard failure in a user-facing system.
- LangGraph merges each node's return dictionary into the running state: only the keys present in the returned dict are updated. This is why the function returns only the three draft fields and the updated `audit_log`, leaving all other state fields unchanged.
- Each audit entry is appended as `state.get("audit_log", []) + [new_entry]`, producing a new list without mutating the existing one.

### Present the draft for approval
With a draft ready, the next node pauses execution and presents the proposal to a human reviewer. In a production system this would mean posting to a Slack channel, opening a ticket, or updating a review dashboard. The critical output of this node is not the display itself - it is the timestamp marking when the review window opened and the status transitioning to `"pending"`.

In [5]:
def present_for_approval(state: ApprovalState) -> dict:
    """Display the draft to the reviewer and open the review window.

    In production this node would dispatch a notification (Slack, email, JIRA ticket) and return immediately, allowing the graph to pause until an external event resumes it. Here we print to stdout as a substitute.
    """
    requested_at = datetime.now()  # Mark the start of the review window

    # Build a compact review card that surfaces everything the approver needs
    print("\n" + "=" * 65)
    print("  APPROVAL REQUEST")
    print("=" * 65)
    print(f"  To      : {state['customer_email']}")
    print(f"  Subject : {state['draft_subject']}")
    print(f"  Timeout : {state['timeout_minutes']} min from now")
    print("-" * 65)
    print("  AGENT REASONING")
    print(f"  {state['draft_reasoning']}")
    print("-" * 65)
    print("  EMAIL BODY")
    print(f"  {state['draft_body']}")
    print("=" * 65)

    audit_entry = {
        "timestamp": requested_at.isoformat(),
        "action": "approval_requested",
        "details": "Draft submitted for human review"
    }

    return {
        "approval_status": "pending",           # Status transitions to pending
        "approval_requested_at": requested_at,  # Marks the start of the review window
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

`present_for_approval` transitions `approval_status` from `None` to `"pending"` and records the `approval_requested_at` timestamp that starts the review window.
- The node deliberately does not collect the human's decision - that responsibility belongs to the next node, keeping each function's scope single and clear.
- In production, this node would dispatch a notification (Slack, email, JIRA) and return immediately. The graph would then pause until an external event resumed it - here we print to stdout as a substitute for that notification mechanism.
- The `approval_requested_at` value is used in the following node to compute elapsed time. If the reviewer took longer than `timeout_minutes`, the next node short-circuits to an automatic timeout response without prompting for any input.

### Collect the human decision
The most sensitive node in the workflow is the one that captures the decision. In a real system this means resuming execution when an external event arrives - a button click in a dashboard, a webhook, or a message on a queue. For this notebook we use Python's `input()` to simulate that event. The node checks elapsed time first, so reviewers who respond after the deadline receive an automatic timeout response regardless of what they type.

In [6]:
def collect_human_decision(state: ApprovalState) -> dict:
    """Collect the reviewer's decision and check it against the timeout.

    Simulates an external approval event using input(). In production this logic would be replaced by polling an API or by LangGraph's interrupt() + MemorySaver pattern, which persists graph state and allows resumption hours later from a separate interface.
    """
    # Compute how much time has elapsed since the review was requested
    elapsed = datetime.now() - state["approval_requested_at"]
    timeout = timedelta(minutes=state["timeout_minutes"])

    # Auto-expire if the review window has already closed — no human input needed
    if elapsed > timeout:
        decided_at = datetime.now()
        audit_entry = {
            "timestamp": decided_at.isoformat(),
            "action": "approval_timeout",
            "details": f"No decision received within {state['timeout_minutes']} min"
        }
        return {
            "approval_status": "timeout",
            "approval_decided_at": decided_at,
            "audit_log": state.get("audit_log", []) + [audit_entry]
        }

    # Collect the reviewer's decision interactively
    print("\nAwaiting your decision...\n")
    raw = input("  Approve this email? (yes / no): ").strip().lower()
    notes = input("  Notes for the audit log (press Enter to skip): ").strip()

    # Any non-"yes" response is treated as rejection — ambiguity must never approve
    status = "approved" if raw == "yes" else "rejected"
    decided_at = datetime.now()

    # Record how long the review took — useful for process bottleneck analysis
    review_seconds = (decided_at - state["approval_requested_at"]).total_seconds()

    audit_entry = {
        "timestamp": decided_at.isoformat(),
        "action": f"approval_{status}",
        "details": (
            f"Decision: {status} | "
            f"Review time: {review_seconds:.1f}s | "
            f"Notes: {notes or 'none'}"
        ),
        "approver": "interactive_user"  # In production: authenticated user ID
    }

    return {
        "approval_status": status,
        "approval_decided_at": decided_at,
        "approver_name": "Demo Reviewer",
        "approval_notes": notes or "No notes provided",
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

`collect_human_decision` does two things in sequence: it checks the timeout, and if the window is still open, it collects the decision interactively.
- The elapsed time comparison (`datetime.now() - state["approval_requested_at"] > timedelta(minutes=...)`) runs before any call to `input()`, so expired reviews are handled instantly without blocking on user input.
- Any input other than the exact string `"yes"` maps to `"rejected"`. This is a deliberate safety asymmetry - an accidental keystroke or ambiguous response must never result in an email being sent.
- The `review_seconds` metric records how long the reviewer took to decide. Aggregated across many runs, this reveals whether approvals are being rubber-stamped too quickly to provide meaningful oversight, or whether the review process is creating bottlenecks.
- In production, `input()` would be replaced by LangGraph's `interrupt()` function combined with a checkpointer (`MemorySaver` or database-backed), which persists graph state and allows resumption hours later from a separate interface.

### Handle the decision outcome
With the decision captured in state, the workflow needs two terminal paths - one that carries out the approved action and one that records the outcome without any side effects. Both functions are kept short because the significant logic was handled upstream in the approval nodes.

In [7]:
def execute_approved_action(state: ApprovalState) -> dict:
    """Send the email — only reachable after an explicit 'approved' status.

    The defensive guard prevents accidental execution if routing logic ever produces an unexpected path to this node.
    """
    # Safety guard: halt if somehow called without a confirmed approval
    if state["approval_status"] != "approved":
        raise ValueError(
            f"execute_approved_action called with status={state['approval_status']}"
        )

    print(f"\n\u2705  APPROVED \u2014 sending email to {state['customer_email']} ...")
    print(f"    Subject : {state['draft_subject']}")

    # --- Replace this block with a real email API in production ---
    # e.g. SendGrid: sg.send(to=state["customer_email"], subject=..., body=...)
    email_sent = True  # Simulated success
    # ---------------------------------------------------------------

    print("    Status  : Sent \u2713")

    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "email_sent",
        "details": f"Email delivered to {state['customer_email']}"
    }

    return {
        "email_sent": email_sent,
        "final_status": "completed_successfully",
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

In [8]:
def handle_rejection(state: ApprovalState) -> dict:
    """Record and surface a rejection or timeout without sending the email."""
    # Distinguish a deliberate rejection from an expired review window
    is_timeout = state["approval_status"] == "timeout"
    label = "TIMEOUT" if is_timeout else "REJECTED"

    print(f"\n\u274c  {label} \u2014 email will not be sent")
    if not is_timeout:
        print(f"    Rejected by : {state.get('approver_name', 'Unknown')}")
        print(f"    Reason      : {state.get('approval_notes', 'None given')}")

    # In production: notify the requester, archive the draft for revision, or trigger an escalation workflow if the timeout was unintended.

    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "rejection_handled",
        "details": (
            f"Status: {state['approval_status']} | "
            f"Notes: {state.get('approval_notes', 'N/A')}"
        )
    }

    return {
        "email_sent": False,
        # Encode the specific reason into final_status for clear audit queries
        "final_status": f"ended_with_{state['approval_status']}",
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

`execute_approved_action` and `handle_rejection` are intentionally short. Their job is to produce the side effect (or the deliberate absence of one) and record the outcome in state.
- The defensive guard in `execute_approved_action` prevents accidental execution if routing logic ever produces an unexpected path to this node. When the stakes are as high as sending a customer email, a redundant check costs nothing.
- `final_status` uses descriptive strings like `"completed_successfully"` and `"ended_with_rejected"`. The rejection string incorporates `state['approval_status']` directly, so rejection and timeout produce distinct queryable values (`"ended_with_rejected"` vs `"ended_with_timeout"`).
- `handle_rejection` distinguishes a deliberate rejection from an expired timeout using the `is_timeout` flag, displaying appropriate messaging for each case without duplicating any function logic.

### Build the approval workflow graph
All five node functions are now defined. The next step is connecting them into a `StateGraph`, establishing the linear sequence from generation to decision, and adding the conditional edge that branches execution based on the approval outcome.

In [9]:
def route_on_approval(state: ApprovalState) -> str:
    """Return the next node name based on the approval outcome.

    This is the sole enforcement point for the approval requirement.
    Any status other than 'approved' routes to rejection handling.
    """
    if state.get("approval_status") == "approved":
        return "execute_action"
    return "handle_rejection"


# Build the graph
workflow = StateGraph(ApprovalState)

# Register each node under the name used by edges and the routing function
workflow.add_node("generate_draft",    generate_email_draft)
workflow.add_node("present_approval",  present_for_approval)
workflow.add_node("collect_decision",  collect_human_decision)
workflow.add_node("execute_action",    execute_approved_action)
workflow.add_node("handle_rejection",  handle_rejection)

# Linear path: generation → display → collect decision
workflow.set_entry_point("generate_draft")
workflow.add_edge("generate_draft",   "present_approval")
workflow.add_edge("present_approval", "collect_decision")

# Conditional branch: approved → execute, anything else → reject
workflow.add_conditional_edges(
    "collect_decision",
    route_on_approval,
    {
        "execute_action":   "execute_action",
        "handle_rejection": "handle_rejection"
    }
)

# Both terminal nodes connect to END
workflow.add_edge("execute_action",  END)
workflow.add_edge("handle_rejection", END)

# Compile validates graph structure and returns an executable object
approval_app = workflow.compile()

The `StateGraph` is built in three phases, each with a distinct purpose.
1. Node registration - each function is bound to a string name that is referenced by edges and the routing function. A mismatch between a registered name and an edge target raises a compile-time error.
2. Edge definitions - three unconditional `add_edge` calls form the linear setup path. One `add_conditional_edges` call implements the approval fork - it calls `route_on_approval` with the full state and routes based on the returned string, which must exactly match one key in the mapping dictionary.
3. Compilation - `workflow.compile()` validates the graph structure before returning an executable. It checks that every referenced node is registered, that there are no unreachable dead ends, and that conditional edge keys are consistent.

`route_on_approval` is defined as a named function rather than a lambda so the routing logic is readable in isolation and straightforward to unit-test without running the full graph.

### Run an example
With the graph compiled, we can run the full workflow end to end. The initial state supplies the customer context and seeds all empty fields. All approval-related fields start as `None`; they will be populated as execution passes through each node.

In [10]:
initial_state: ApprovalState = {
    "customer_request": "I need help resetting my password. I've been locked out for two days.",
    "customer_email": "customer@example.com",
    # Draft fields are None here; generate_email_draft will populate them
    "draft_subject": None,
    "draft_body": None,
    "draft_reasoning": None,
    # Approval fields are None here; the approval nodes will populate them
    "approval_status": None,
    "approval_requested_at": None,
    "approval_decided_at": None,
    "approver_name": None,
    "approval_notes": None,
    "timeout_minutes": 30,  # 30-minute review window before auto-timeout
    # Outcome fields are populated by the terminal node
    "email_sent": False,
    "final_status": None,
    "audit_log": []
}

print("\U0001f680  Running approval workflow...\n")
result = approval_app.invoke(initial_state)

print("\n" + "=" * 65)
print("  WORKFLOW SUMMARY")
print("=" * 65)
print(f"  Final status    : {result['final_status']}")
print(f"  Email sent      : {result['email_sent']}")
print(f"  Approval status : {result['approval_status']}")
print(f"  Decided by      : {result.get('approver_name', 'N/A')}")
print(f"  Notes           : {result.get('approval_notes', 'N/A')}")

🚀  Running approval workflow...


  APPROVAL REQUEST
  To      : customer@example.com
  Subject : Assistance with Your Password Reset Request
  Timeout : 30 min from now
-----------------------------------------------------------------
  AGENT REASONING
  In crafting the response, I chose a friendly and empathetic tone to convey understanding of the customer's frustration regarding being locked out for two days. The key points addressed include acknowledging the inconvenience, providing clear instructions for resetting the password, and assuring the customer of our support throughout the process. Risks to be aware of include ensuring the instructions are accurate to avoid any further frustration, and being sensitive to the customer's potential anxiety about account access.
-----------------------------------------------------------------
  EMAIL BODY
  Dear Customer,

Thank you for reaching out to us regarding your password reset. I’m sorry to hear that you’ve been locked out of your a

  Approve this email? (yes / no):  yes
  Notes for the audit log (press Enter to skip):  Correct reference



✅  APPROVED — sending email to customer@example.com ...
    Subject : Assistance with Your Password Reset Request
    Status  : Sent ✓

  WORKFLOW SUMMARY
  Final status    : completed_successfully
  Email sent      : True
  Approval status : approved
  Decided by      : Demo Reviewer
  Notes           : Correct reference


In [11]:
# Display the complete audit trail produced by this run
print("\n" + "=" * 65)
print("  AUDIT TRAIL")
print("=" * 65)

for idx, entry in enumerate(result["audit_log"], start=1):
    print(f"\n  [{idx}] {entry['action'].upper()}")
    print(f"       Timestamp : {entry['timestamp']}")
    print(f"       Detail    : {entry['details']}")
    if "approver" in entry:  # Only approval decisions include an approver field
        print(f"       By        : {entry['approver']}")


  AUDIT TRAIL

  [1] DRAFT_GENERATED
       Timestamp : 2026-03-15T09:27:11.296038
       Detail    : Subject: Assistance with Your Password Reset Request

  [2] APPROVAL_REQUESTED
       Timestamp : 2026-03-15T09:27:11.296586
       Detail    : Draft submitted for human review

  [3] APPROVAL_APPROVED
       Timestamp : 2026-03-15T09:27:46.143383
       Detail    : Decision: approved | Review time: 34.8s | Notes: Correct reference
       By        : interactive_user

  [4] EMAIL_SENT
       Timestamp : 2026-03-15T09:27:46.144058
       Detail    : Email delivered to customer@example.com


Each audit entry corresponds to exactly one node execution and appears in the order nodes ran. The `action` field uses a consistent naming scheme - `draft_generated`, `approval_requested`, `approval_approved`/`approval_rejected`, `email_sent` - which makes the log filterable by event type for compliance queries. The difference between the `approval_requested` and `approval_decided` timestamps gives the reviewer response time, a metric that reveals whether the review process is creating bottlenecks or whether approvals are being made too quickly to be meaningful.

## Multi-level approval hierarchy
The single-level workflow works well when all emails carry similar risk. In practice, risk varies significantly: a password-reset acknowledgement carries very different stakes than an email granting a refund or making a policy exception. A multi-level approval hierarchy routes each draft through the number of review levels its risk warrants - low-risk drafts get a single quick approval while high-risk drafts climb an authority chain before sending.

### Define the multi-level state
The hierarchy introduces two new concerns not present in the single-level state: a risk classification that determines how many levels are required, and a loop counter that tracks progress through the chain. We define a separate `TypedDict` to keep these concerns cleanly separated from the single-level implementation.

In [12]:
class MultiLevelApprovalState(TypedDict):
    # Customer context and draft fields — same structure as the single-level workflow
    customer_request: str
    customer_email: str
    draft_subject: Optional[str]
    draft_body: Optional[str]
    draft_reasoning: Optional[str]

    # Risk classification controls how many approval levels are required
    risk_level: Optional[Literal["low", "medium", "high"]]
    levels_required: Optional[int]  # 1 (low), 2 (medium), or 3 (high)

    # Loop counter — starts at 0 before the chain begins, increments each pass
    current_level: int

    # One record per completed approval level; grows monotonically through the chain
    approval_chain: List[dict]

    # Outcome
    email_sent: bool
    final_status: Optional[str]  # None while in-progress; set when chain ends
    audit_log: List[dict]

`MultiLevelApprovalState` replaces the single `approval_status` field with a richer structure built around the concepts of chain length and progress tracking.
- `levels_required` is set by the risk assessment node and controls how many iterations the approval loop runs.
- `current_level` is the loop counter. It starts at zero before the chain begins and increments on every pass through the approval node.
- `approval_chain` accumulates one decision record per level, growing monotonically through the chain.
- `final_status` is the loop termination signal. It starts as `None` while the chain is in progress, transitions to `"fully_approved"` when the last level approves, and transitions to `"rejected_at_level_N"` when any level rejects. The routing function reads this field to decide whether to loop, execute, or reject.

### Assess the draft's risk level
Before entering the approval chain, the workflow needs to classify the draft's risk. Rather than using hard-coded keyword rules - which are brittle and miss paraphrases - we ask the LLM to evaluate the full draft holistically. The classification directly controls the chain length, making it a consequential decision rather than a metadata tag.

In [13]:
def assess_risk(state: MultiLevelApprovalState) -> dict:
    """Classify the draft's risk level and set the required number of approval levels."""
    prompt = f"""Rate the risk of sending this customer email.

Customer request: {state['customer_request']}
Email subject: {state['draft_subject']}
Email body: {state['draft_body']}

Risk classification:
- LOW    : Informational replies, standard responses, no commitments
- MEDIUM : Account changes, minor commitments, mention of credits or refunds
- HIGH   : Large financial commitments, policy exceptions, legal language, PII

Reply with exactly one word: LOW, MEDIUM, or HIGH."""

    response = llm.invoke(prompt)

    # The model may add punctuation; scan for each label in severity order
    raw = response.content.strip().upper()
    risk_level = "medium"  # Safe default if none of the labels appear
    for label in ("HIGH", "MEDIUM", "LOW"):
        if label in raw:
            risk_level = label.lower()
            break

    # Map the risk label to the number of approval levels required
    levels_map = {"low": 1, "medium": 2, "high": 3}
    levels_required = levels_map[risk_level]

    print(f"\n\U0001f3af  Risk: {risk_level.upper()}  \u2192  {levels_required} approval level(s) required")

    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "risk_assessed",
        "details": f"Risk: {risk_level} | Levels required: {levels_required}"
    }

    return {
        "risk_level": risk_level,
        "levels_required": levels_required,
        "current_level": 0,      # Reset the counter before the loop starts
        "approval_chain": [],    # Ensure the chain is empty before the first pass
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

`assess_risk` uses the LLM for classification, then converts the label to a required level count stored in state.
- The parsing loop checks for each label in descending severity order (`HIGH` before `MEDIUM` before `LOW`). Scanning in this order ensures that a response like `"HIGH RISK"` correctly resolves to `"high"` rather than accidentally matching a shorter label.
- Setting `current_level: 0` and `approval_chain: []` explicitly in this node guarantees the counters are clean even if the state was carried over from a previous run.
- The `"medium"` default when no label matches is a conservative choice - if the LLM fails to classify, the system applies a middle-tier approval requirement rather than the lowest overhead.

### Iterate through the approval chain
With the risk level set, the workflow enters the approval loop. Each pass through the approval node requests sign-off from the authority at the current level and then returns one of three signals: rejection (chain ends immediately), full approval (all levels cleared), or continuation (more levels remain). LangGraph supports routing a node back to itself through a conditional edge, which is how we implement the loop without duplicating any node logic.

In [14]:
def request_level_approval(state: MultiLevelApprovalState) -> dict:
    """Request approval from the reviewer at the current authority level.

    Authority levels:
      1 -> Team Lead   (basic content review)
      2 -> Manager     (business impact review)
      3 -> Director    (strategic and risk review)

    A rejection at any level terminates the chain immediately.
    Returning final_status=None signals the routing function to loop back.
    """
    # Advance to the next level before collecting input
    level = state["current_level"] + 1
    role_map = {1: "Team Lead", 2: "Manager", 3: "Director"}
    role = role_map.get(level, f"Approver L{level}")

    print(f"\n\U0001f4cb  Level {level} / {state['levels_required']} \u2014 {role}")
    print(f"    Subject : {state['draft_subject']}")

    raw = input(f"  {role}: Approve? (yes / no): ").strip().lower()
    notes = input("  Notes: ").strip()

    # Map input to decision; non-yes is always rejection
    decision = "approved" if raw == "yes" else "rejected"

    # Append this level's outcome to the growing chain record
    chain_entry = {
        "level": level,
        "role": role,
        "decision": decision,
        "notes": notes or "none",
        "timestamp": datetime.now().isoformat()
    }
    updated_chain = state.get("approval_chain", []) + [chain_entry]

    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": f"level_{level}_{decision}",
        "details": f"{role}: {decision} | Notes: {notes or 'none'}"
    }

    if decision == "rejected":
        # Any rejection terminates the chain — set final_status and stop looping
        return {
            "current_level": level,
            "approval_chain": updated_chain,
            "final_status": f"rejected_at_level_{level}",
            "email_sent": False,
            "audit_log": state.get("audit_log", []) + [audit_entry]
        }

    if level >= state["levels_required"]:
        # All required levels have approved — mark the chain as complete
        return {
            "current_level": level,
            "approval_chain": updated_chain,
            "final_status": "fully_approved",
            "email_sent": False,  # The execute node will set this to True
            "audit_log": state.get("audit_log", []) + [audit_entry]
        }

    # More levels remain — return final_status=None to signal another loop iteration
    return {
        "current_level": level,
        "approval_chain": updated_chain,
        "final_status": None,
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }

`request_level_approval` has three return paths, each setting `final_status` to a different value that the routing function reads.
- A rejection sets `final_status` to `"rejected_at_level_N"` and the chain terminates immediately - no subsequent levels are consulted, and `email_sent` is set to `False`.
- Clearing the last required level sets `final_status` to `"fully_approved"`. The node still sets `email_sent: False` here, ensuring the send action is attributed to the execute node in the audit log, not to the approval step.
- Any other outcome leaves `final_status` as `None`, which is the signal the routing function reads to loop back for the next level.

### Build and test the multi-level workflow
With the approval loop node defined, we wire the multi-level graph together, add wrapper execute and reject nodes adapted to `MultiLevelApprovalState`, and run a test on a high-stakes refund request. The loop-back edge - `"request_level" → "request_level"` - is the key structural feature that implements the multi-level chain without any code duplication.

In [15]:
def route_multi_level(state: MultiLevelApprovalState) -> str:
    """Route after each approval level iteration.

    Returns:
      'request_level'   -- final_status is None, more levels needed (loop)
      'execute_action'  -- all levels approved
      'handle_rejection' -- rejected at this level
    """
    status = state.get("final_status")
    if status == "fully_approved":
        return "execute_action"
    if status and status.startswith("rejected_at_level"):
        return "handle_rejection"
    # final_status is None: the chain is incomplete, loop back
    return "request_level"


# Thin wrappers adapt the shared terminal logic to MultiLevelApprovalState
def ml_execute(state: MultiLevelApprovalState) -> dict:
    """Send the email after the full approval chain is satisfied."""
    print(f"\n\u2705  ALL LEVELS APPROVED \u2014 sending email to {state['customer_email']}")
    # Show who approved at each level for immediate accountability
    roles = [e["role"] for e in state["approval_chain"]]
    print(f"    Chain: {' \u2192 '.join(roles)}")
    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "email_sent",
        "details": f"Sent after {state['levels_required']}-level approval"
    }
    return {
        "email_sent": True,
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }


def ml_reject(state: MultiLevelApprovalState) -> dict:
    """Record rejection from any level in the chain."""
    print(f"\n\u274c  CHAIN REJECTED \u2014 email will not be sent ({state['final_status']})")
    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "action": "chain_rejected",
        "details": state["final_status"]
    }
    return {
        "email_sent": False,
        "audit_log": state.get("audit_log", []) + [audit_entry]
    }


# Build the multi-level graph; generate_email_draft is reused from the first workflow
ml_workflow = StateGraph(MultiLevelApprovalState)

ml_workflow.add_node("generate_draft",    generate_email_draft)
ml_workflow.add_node("assess_risk",       assess_risk)
ml_workflow.add_node("request_level",     request_level_approval)
ml_workflow.add_node("execute_action",    ml_execute)
ml_workflow.add_node("handle_rejection",  ml_reject)

ml_workflow.set_entry_point("generate_draft")
ml_workflow.add_edge("generate_draft", "assess_risk")
ml_workflow.add_edge("assess_risk",    "request_level")

# The 'request_level' -> 'request_level' edge is the loop-back
ml_workflow.add_conditional_edges(
    "request_level",
    route_multi_level,
    {
        "request_level":    "request_level",    # Loop back for next level
        "execute_action":   "execute_action",
        "handle_rejection": "handle_rejection"
    }
)

ml_workflow.add_edge("execute_action",   END)
ml_workflow.add_edge("handle_rejection", END)

ml_app = ml_workflow.compile()
print("Multi-level approval workflow compiled")

Multi-level approval workflow compiled


In [16]:
# Test with a refund request — likely to be classified as medium or high risk
ml_initial: MultiLevelApprovalState = {
    "customer_request": (
        "I want a full refund of $450 for a service I purchased last month. "
        "The product did not work as described and I want this resolved today."
    ),
    "customer_email": "refund_customer@example.com",
    # Draft fields — populated by generate_email_draft
    "draft_subject": None,
    "draft_body": None,
    "draft_reasoning": None,
    # Risk and chain fields — populated by assess_risk and request_level_approval
    "risk_level": None,
    "levels_required": None,
    "current_level": 0,
    "approval_chain": [],
    # Outcome fields
    "email_sent": False,
    "final_status": None,
    "audit_log": []
}

print("\U0001f680  Running multi-level approval workflow...\n")
ml_result = ml_app.invoke(ml_initial)

print("\n" + "=" * 65)
print("  MULTI-LEVEL WORKFLOW SUMMARY")
print("=" * 65)
print(f"  Risk level      : {ml_result.get('risk_level', 'N/A').upper()}")
print(f"  Levels required : {ml_result.get('levels_required', 'N/A')}")
print(f"  Email sent      : {ml_result['email_sent']}")
print(f"  Final status    : {ml_result['final_status']}")

print("\n  APPROVAL CHAIN")
for entry in ml_result.get("approval_chain", []):
    icon = "\u2713" if entry["decision"] == "approved" else "\u2717"
    print(f"  {icon}  Level {entry['level']} ({entry['role']}): {entry['decision']} \u2014 {entry['notes']}")

print("\n  AUDIT LOG")
for idx, entry in enumerate(ml_result["audit_log"], start=1):
    print(f"  [{idx}] {entry['action']} | {entry['details']}")

🚀  Running multi-level approval workflow...


🎯  Risk: MEDIUM  →  2 approval level(s) required

📋  Level 1 / 2 — Team Lead
    Subject : Request for Refund - Order Confirmation Needed


  Team Lead: Approve? (yes / no):  yes
  Notes:  Customer is not happy



📋  Level 2 / 2 — Manager
    Subject : Request for Refund - Order Confirmation Needed


  Manager: Approve? (yes / no):  no
  Notes:  Not according to the contract conditions



❌  CHAIN REJECTED — email will not be sent (rejected_at_level_2)

  MULTI-LEVEL WORKFLOW SUMMARY
  Risk level      : MEDIUM
  Levels required : 2
  Email sent      : False
  Final status    : rejected_at_level_2

  APPROVAL CHAIN
  ✓  Level 1 (Team Lead): approved — Customer is not happy
  ✗  Level 2 (Manager): rejected — Not according to the contract conditions

  AUDIT LOG
  [1] draft_generated | Subject: Request for Refund - Order Confirmation Needed
  [2] risk_assessed | Risk: medium | Levels required: 2
  [3] level_1_approved | Team Lead: approved | Notes: Customer is not happy
  [4] level_2_rejected | Manager: rejected | Notes: Not according to the contract conditions
  [5] chain_rejected | rejected_at_level_2


The two workflows built in this notebook show how approval gates integrate cleanly into an agent's execution pipeline without requiring the agent's core generation logic to be redesigned around the approval requirement.

When to use approval workflows:
- Actions that are irreversible once executed, such as sent emails or financial transactions.
- Actions with compliance obligations like regulated communications or data sharing.
- Actions whose risk varies enough that blanket human review would create approval fatigue.

Design principles that matter in production:
- Surface the agent's reasoning alongside the proposal. Reviewers who see only the output cannot efficiently judge whether the intent was appropriate.
- Set a reasonable timeout and route expired reviews to rejection or re-escalation. A workflow that waits indefinitely for human input is not production-ready.
- Keep each node's responsibility narrow - one node opens the review window, a separate node collects the decision, and separate terminal nodes handle execution and rejection.

Multi-level hierarchies ensure approval overhead is proportional to actual risk. Trivially safe drafts bypass unnecessary review steps; high-stakes drafts climb the authority chain. The loop-back conditional edge in LangGraph makes this pattern easy to express without duplicating node logic.

Audit trails in production should be written to an external append-only store - a database, object storage, or log aggregation service - rather than living in the workflow state, so they persist beyond the lifecycle of any individual run and remain available for compliance reporting independent of the workflow engine.